# Intrinsic Benchmarks - Information Density

This notebook computes and visualizes the information density - intrinsic benchmarking metrics for the connectivity matrices of each of the PySPI and skarf methods.

Averages runs per subject first via `load_avg_mats_and_impose_sparsity`,
then computes all metrics on the averaged matrix.

In [ ]:
import os
for var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[var] = "1"

In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import json

PROJECT_ROOT = Path("/home/jpillai/projects/skarf-experiments")

from arfcexp.matrices import EfficientMatrixReader, load_avg_mats_and_impose_sparsity
from arfcexp.info_density import compute_all_intrinsic

# paths
PARQUET_PATH = Path("/srv/projects/skarf/data_aggregation/hcp_1200_rfmri_schaefer.parquet")
SUBJECT_LIST = PROJECT_ROOT / "resources/subject_lists/hcp_complete_data_867_subject_list.txt"
OUT_DIR = PROJECT_ROOT / "results/intrinsic_benchmarks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# run params
SPARSITY = 0.0       
TSP_NODES = list(range(20))
SMALL_WORLD_KWARGS = dict(seed=70, nrand=1, nswap=2000)
RICH_CLUB_KWARGS = {"normalized": False}

# subjects
with open(SUBJECT_LIST) as f:
    SUB_LIST = [line.strip() for line in f if line.strip()]

def load_lookup(project_root, filename):
    with open(project_root / "resources" / filename) as f:
        return json.load(f)

SYMMETRY_LOOKUP = load_lookup(PROJECT_ROOT, "matrix_symmetry_lookup.json")
DEGENERATE_LOOKUP = load_lookup(PROJECT_ROOT, "matrix_degenerate_lookup.json")
ANTISYMMETRY_LOOKUP = load_lookup(PROJECT_ROOT, "matrix_antisymmetry_lookup.json")

# create reader once (index is built on first query and reused across all combos)
READER = EfficientMatrixReader(PARQUET_PATH)

print(f"Subjects: {len(SUB_LIST)}")
print(f"Output: {OUT_DIR}")

In [ ]:
schema = pl.scan_parquet(PARQUET_PATH).schema
has_lag = "lag" in schema

select_cols = ["method", "func"] + (["lag"] if has_lag else [])
combos = (
    pl.scan_parquet(PARQUET_PATH)
    .filter(pl.col("success"))
    .select(select_cols)
    .unique()
    .collect()
    .to_pandas()
)
if not has_lag:
    combos["lag"] = None

combos = combos.reset_index(drop=True)
print(f"{len(combos)} (method, func, lag) combinations found")
combos

In [ ]:
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

def process_density_combo(combo_idx):
    # Move reader creation INSIDE the function — each worker gets its own
    reader = EfficientMatrixReader(PARQUET_PATH)
    
    with threadpool_limits(limits=1):
        row = combos.iloc[combo_idx]
        method = row["method"]
        func = row["func"]
        lag = row["lag"]
        lag_val = None if pd.isna(lag) else int(lag)
        key = f"{method}__{func}"

        print(f"\n[{combo_idx+1}/{len(combos)}] Starting {method}/{func}/lag={lag}", flush=True)

        if DEGENERATE_LOOKUP.get(key, False):
            print(f"  -> SKIP (degenerate)", flush=True)
            return pd.DataFrame()

        is_symmetric = SYMMETRY_LOOKUP.get(key, False)
        is_antisymmetric = ANTISYMMETRY_LOOKUP.get(key, False)
        if is_antisymmetric:
            is_symmetric = False

        avg_df = load_avg_mats_and_impose_sparsity(
            PARQUET_PATH,
            method=method,
            func=func,
            sub_list=SUB_LIST,
            sparsity=SPARSITY,
            symmetry_lookup=SYMMETRY_LOOKUP,
            lag=lag_val or 0,
            reader=reader,  # use local reader
        )

        valid = avg_df[avg_df["Matrix"].notna() & (avg_df["Count"] > 0)]
        n_valid = len(valid)
        if valid.empty:
            print(f"  -> No valid subjects, skipping", flush=True)
            return pd.DataFrame()

        print(f"  -> {n_valid} subjects to process", flush=True)

        rows = []
        for i, (sub, row_data) in enumerate(valid.iterrows(), 1):
            if i % 100 == 0 or i == n_valid:
                print(f"  -> {i}/{n_valid} (id={sub})", flush=True)
            try:
                metrics = compute_all_intrinsic(
                    row_data["Matrix"],
                    is_symmetric=is_symmetric,
                    tsp_nodes=TSP_NODES,
                    small_world_kwargs=SMALL_WORLD_KWARGS,
                    rich_club_kwargs=RICH_CLUB_KWARGS,
                )
            except Exception as exc:
                print(f"  [WARN] sub={sub} {method}/{func}/lag={lag}: {exc}", flush=True)
                metrics = {}
            rows.append({"sub": sub, "method": method, "func": func, "lag": lag, **metrics})

        print(f"  -> Done: {len(rows)} rows", flush=True)
        return pd.DataFrame(rows)

In [ ]:
N_JOBS = 4 

all_dfs = []
n_combos = len(combos)
print(f"Starting processing: {n_combos} combos × up to {len(SUB_LIST)} subjects, {N_JOBS} parallel workers\n")

results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=0)(
    delayed(process_density_combo)(i) for i in range(n_combos)
)

all_dfs = [df for df in results if df is not None and not df.empty]

print(f"\nAll combos done. Concatenating and saving...")
density_df = pd.concat(all_dfs, ignore_index=True)
density_df.to_parquet(OUT_DIR / "info_density.parquet", index=False)
print(f"Done — {len(density_df)} rows saved to {OUT_DIR / 'info_density.parquet'}")